In [1]:
import csv, json, requests, re, ast, copy
import numpy as np
from scipy import spatial
from tqdm import tqdm
from openpyxl import load_workbook
import pandas as pd
import html
import pandas as pd
import re, html, json, ast
import time
from typing import List, Dict, Any, Tuple,Optional

In [2]:
#used for decision tree generation
import xml.etree.ElementTree as ET
from xml.dom import minidom
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import warnings
warnings.filterwarnings("ignore")

In [3]:
from langchain_openai import AzureChatOpenAI

############## Embedding Creation
from openai import AzureOpenAI

#from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

#llm = ChatOpenAI(model="gpt-4o", temperature=0, api_key=api_key)

############ Used for chaining
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
#from langchain_core.runnables.base import RunnableSequence


In [4]:
api_key="4d78888ae7d34d38bf5a0ec97a69a6f6"

In [5]:
# LLM Setup
llm = AzureChatOpenAI(
    api_key=api_key,
    azure_endpoint="https://genaitraining-aoai2.openai.azure.com/",
    azure_deployment="gpt-4o",  # This must match your Azure deployment name
    api_version="2024-12-01-preview",  # Or "2025-01-01-preview" if that's correct
    temperature=0
)

def get_completion(prompt, model=llm):
    response = model.invoke([HumanMessage(content=prompt)])
    return response.content


client = AzureOpenAI(
  api_key = api_key,  
  api_version = "2024-10-21",
  azure_endpoint ="https://genaitraining-aoai2.openai.azure.com/"
)

def get_embedding_vector(text):
    embedding_response = client.embeddings.create(
        input = text,
        #model= "text-embedding-3-large"
        model= "text-embedding-ada-002"
    )

    embedding_response_dict = embedding_response.model_dump_json()
    parsed_embedding_respomde_dict = json.loads(embedding_response_dict)
    embedding_vector = parsed_embedding_respomde_dict['data'][0]['embedding']

    return embedding_vector


In [6]:
def write_lists_csv(list1,list2,header1,header2, output_filename):
   
    with open(output_filename, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow([header1, header2])  # Write header row
        max_length = max(len(list1), len(list2))
        for i in range(max_length):
            row = [list1[i] if i < len(list1) else '', list2[i] if i < len(list2) else '']
            writer.writerow(row)  # Write data rows
    return

In [7]:
def extract_json_string(text: str):
    # 1) Trim and strip code fences (``` or ```json)
    t = text.strip()
    t = re.sub(r'^\s*```(?:json)?\s*|\s*```\s*$', '', t, flags=re.IGNORECASE)

    # 2) If the entire payload is a quoted Python string literal, unquote it safely.
    #    This handles cases like the one in text.txt: '[\\n  { ... }\\n]'
    if (t.startswith("'") and t.endswith("'")) or (t.startswith('"') and t.endswith('"')):
        try:
            t = ast.literal_eval(t)
        except Exception:
            # If it's not a valid Python string literal, continue with t as-is
            pass

    # 3) Decode HTML entities (&amp;, &lt;, ...)
    t = html.unescape(t)

    # 4) Normalize Python-ish literals to JSON
    t = t.replace("True", "true").replace("False", "false").replace("None", "null")

    # IMPORTANT: Do NOT do `.encode().decode('unicode_escape')` here.
    # That would turn \\n into literal newlines and break JSON,
    # and also produce invalid escapes like \(, which JSON doesn't allow.

    # 5) Try parsing as JSON
    try:
        return json.loads(t)
    except json.JSONDecodeError:
        # 6) Fallback: extract the first top-level array/object and parse that
        m = re.search(r'(\[.*\]|\{.*\})', t, flags=re.DOTALL)
        if m:
            return json.loads(m.group(1))
        # Optional: try a permissive library if available (uncomment if you use demjson3)
        # import demjson3
        # return demjson3.decode(t)
        raise


In [8]:
def read_lcd_from_excelfile(lcd_id, excelfile_path = './LCDs/lcd.xlsx'):
    # Load the workbook
    workbook = load_workbook(filename=excelfile_path)
    
    # Select a specific sheet
    sheet = workbook['lcd']  # Change to your desired sheet name

    # Read header row to find column indices
    header = [cell.value for cell in sheet[1]]
    try:
        
        lcd_id_index = header.index('lcd_id')
        indication_index = header.index('indication')
        title_index = header.index('title')

    except ValueError:
        raise Exception("Required columns not found in the header.")

    for row in sheet.iter_rows(min_row=2, values_only=True):  # Skip header
        if row[lcd_id_index] == lcd_id:
            content = row[indication_index]
            procedure_title = row[title_index]
            return content, procedure_title


    else:
        print(f"lcd_id {lcd_id} not found.")

        return None


In [9]:
def fetch_LCD(url):
    response = requests.get(url)
    
    # to avoids encoding errors and replaces any problematic characters with safe placeholders
    text = response.content.decode('utf-8', errors='replace')
    # In order to decrease the size of the text, we will extract only the content between "Coverage Guidance" and "General Information"
    # This will help us focus on the relevant part of the document for further processing and lessen the token count for the LLM.
    pattern = r'Coverage Guidance</h3>(.*?)General Information</h2>'
    match = re.search(pattern, text, re.DOTALL)
    content = match.group(1) if match else None

    # Extract the title of the procedure
    title_pattern = r'<title>LCD\s+-\s+(.*?)\s*\(.*?\)</title>'
    match_title = re.search(title_pattern, text, re.DOTALL)
    LCD_title = html.unescape(match_title.group(1) )if match_title else None

    # Extract the LCD ID
    LCD_ID_pattern = r'<title>LCD\s+-\s+.*?\((.*?)\)</title>'
    LCD_ID_match = re.search(LCD_ID_pattern, text)
    LCD_ID = LCD_ID_match.group(1) if LCD_ID_match else None
    
    return content, LCD_title, LCD_ID

In [10]:
# This cell is used for testing.
#url = 'https://www.cms.gov/medicare-coverage-database/view/lcd.aspx?LCDId=34635' #Botulinum Toxin Type A & Type B
#url = 'https://www.cms.gov/medicare-coverage-database/view/lcd.aspx?LCDId=36573&ContrId=345' #THA
#url = 'https://www.cms.gov/medicare-coverage-database/view/lcd.aspx?LCDId=36575' #TKA
#url = 'https://www.cms.gov/medicare-coverage-database/view/lcd.aspx?lcdid=35172&ver=68&bc=0'
url = 'https://www.cms.gov/medicare-coverage-database/view/lcd.aspx?lcdid=35172&ver=68&bc=0'
#url = 'https://www.cms.gov/medicare-coverage-database/view/lcd.aspx?lcdid=35172&ver=68&bc=0'


LCD, LCD_title, LCD_ID = fetch_LCD(url)
name = LCD_title
print(LCD_title)
print(LCD_ID)

Botulinum Toxin Types A and B
L35172


In [11]:
# Sample URL Need to process data
LCD_URLs = [
'https://www.cms.gov/medicare-coverage-database/view/lcd.aspx?lcdid=35172&ver=68&bc=0',
'https://www.cms.gov/medicare-coverage-database/view/lcd.aspx?lcdid=38809&ver=6&bc=0',
'https://www.cms.gov/medicare-coverage-database/view/lcd.aspx?lcdid=36286&ver=21&bc=0',
'https://www.cms.gov/medicare-coverage-database/view/lcd.aspx?lcdid=34194&ver=23&keyword=L34194&keywordType=starts&areaId=all&docType=NCA,CAL,NCD,MEDCAC,TA,MCD,6,3,5,1,F,P&contractOption=all&sortBy=relevance&bc=1',
'https://www.cms.gov/medicare-coverage-database/view/lcd.aspx?lcdid=35004&ver=23&bc=0',
'https://www.cms.gov/medicare-coverage-database/view/lcd.aspx?lcdid=38803&ver=16&bc=0',
'https://www.cms.gov/medicare-coverage-database/view/lcd.aspx?lcdid=34892&ver=123&bc=0',
'https://www.cms.gov/medicare-coverage-database/view/lcd.aspx?lcdid=38841&ver=16&bc=0',
'https://www.cms.gov/medicare-coverage-database/view/lcd.aspx?lcdid=34010&ver=37&bc=0',
'https://www.cms.gov/medicare-coverage-database/view/lcd.aspx?lcdid=34924&ver=79&bc=0',
'https://www.cms.gov/medicare-coverage-database/view/lcd.aspx?lcdid=35083&ver=108&bc=0',
'https://www.cms.gov/medicare-coverage-database/view/lcd.aspx?lcdid=36204&ver=22&bc=0',
'https://www.cms.gov/medicare-coverage-database/view/lcd.aspx?lcdid=35136&ver=33&bc=0',
'https://www.cms.gov/medicare-coverage-database/view/lcd.aspx?lcdid=33611&ver=25&bc=0', 
'https://www.cms.gov/medicare-coverage-database/view/lcd.aspx?lcdid=33718&ver=52&bc=0 ' 
]

In [12]:
LCD_title, LCD_ID

('Botulinum Toxin Types A and B', 'L35172')

# Leaf Node Function body

In [13]:
def collect_leaf_nodes(decision_tree: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """
    Returns every leaf MDG with:
      - member_id
      - content
      - clinical_context_chain: list of ancestor *contents* ordered
        from closest parent -> root.

    A node is NON-leaf if:
      (a) its member_id is a dotted prefix of another node's id, OR
      (b) its content appears as the clinical_context of any other node.
    """

    def norm(s: str) -> str:
        # Normalize for robust matching (not used for output)
        s = (s or "")
        s = re.sub(r"\s+", " ", s).strip()
        return s[:-1] if s.endswith(":") else s

    # ---- 1) Flatten all mdgs from the nested structure
    all_mdgs = []

    def walk_group(group: Dict[str, Any]):
        for m in (group.get('mdgs') or []):
            all_mdgs.append(m)
        for ch in (group.get('child_group') or []):
            walk_group(ch)

    for item in (decision_tree or []):
        pg = (item.get('parent_group') or {})
        walk_group(pg)

    # Indexes
    id_to_mdg: Dict[str, Dict[str, Any]] = {m['member_id']: m for m in all_mdgs if 'member_id' in m}

    # Map normalized content -> ids
    content_to_ids: Dict[str, List[str]] = {}
    for m in id_to_mdg.values():
        ckey = norm(m.get('content'))
        if ckey:
            content_to_ids.setdefault(ckey, []).append(m['member_id'])

    # ---- 2) Compute NON-leaf ids
    non_leaf_ids = set()

    # (a) Structural parents by dotted prefixes
    for mid in id_to_mdg.keys():
        parts = mid.split('.')
        for k in range(1, len(parts)):
            non_leaf_ids.add('.'.join(parts[:k]))

    # (b) Semantic parents: nodes whose content is used as someone else's clinical_context
    for m in id_to_mdg.values():
        cc_norm = norm(m.get('clinical_context'))
        if cc_norm:
            for parent_id in content_to_ids.get(cc_norm, []):
                non_leaf_ids.add(parent_id)

    # ---- 3) Build leaves + clinical_context_chain
    def unique_preserve_order(seq):
        seen, out = set(), []
        for x in seq:
            if x and x not in seen:
                seen.add(x)
                out.append(x)
        return out

    leaves = []
    for mid, m in id_to_mdg.items():
        if mid in non_leaf_ids:
            continue  # not a leaf

        # Ancestors via dotted prefixes
        parts = mid.split('.')
        ancestor_ids = ['.'.join(parts[:k]) for k in range(1, len(parts))]
        ancestor_ids = [aid for aid in ancestor_ids if aid in id_to_mdg]

        # closest parent -> root, using ORIGINAL ancestor contents
        clinical_context_chain = [
            id_to_mdg[aid].get('content') for aid in reversed(ancestor_ids)
            if id_to_mdg[aid].get('content')
        ]
        clinical_context_chain = unique_preserve_order(clinical_context_chain)

        leaves.append({
            'member_id': mid,
            'content': m.get('content'),
            'clinical_context_chain': clinical_context_chain
        })

    return leaves


# LCD Medical Guideline Extration

In [14]:
def extract_LCD_medicalguidelines(LCD, LCD_title):

  instruct_template = """

    Instruct: You are expert in prior authorization in health insurance companies. Your job is extract all medical guidelines, including indications\

    and limitations, from a Local Coverage Determination (LCD) document. Here is their definition:

      - Indications: These are the clinical scenarios, diagnoses, or patient conditions under which a service, procedure, or item is considered \
          reasonable and necessary and therefore eligible for Medicare coverage.

      - Limitations: These are the boundaries or restrictions placed on the use of a service, even when it is generally covered. They define how \
          often, under what conditions, or for which patient populations the service is not covered.

      Please pay attention we don't need any other information in LCD's beyond indications and limitations. Please keep the definition of indications \
      and limitations for next prompts.
      In continue, I am going to ask you to extract some information from an LCD document which is in html format. Please don't respond to this prompt and wait for next prompts.
  """

  contraindications_extraction_prompt = """
  thought:
    In medicine, a contraindication is a condition (a situation or factor) that serves as a reason not to take a certain medical treatment 
    or procedure due to the harm that it would cause the patient. Contraindication is the opposite of indication, which is a reason to use a 
    certain procedure or treatment, so contraindication is a reason to not use a certain procedure or treatment.

    Contraindications are distinct from limitations. Limitations refer to coverage boundaries such as frequency, dosage, or quantity restrictions, 
    and should not be extracted. Only extract contraindications that relate to clinical safety concerns. Don't consider limitations.

    Contraindications can be classified into two main categories based on how they are presented in the document:
    - "explicit" contraindications: typically presented in structured formats such as:
      - Bullet points (<ul><li>) or (&bull;)
      - Numbered or alphabetically ordered lists (<ol type="1"|"A"|"a">)

    - "implicit" contraindications: refer to those that are not explicitly listed in structured formats such as bullet points, numbered lists, 
      or tables. Instead, they are embedded within narrative paragraphs of the document, often requiring careful interpretation to 
      extract relevant clinical meaning.

    Alongside exrtacting contraindications, it's equally important to identify the **clinical context**.
    Clinical context typically:
        - Provides background, rationale, or framing for contraindications that follow.
        - Appears as plain text or paragraph(s), or as title section before explicit guidelines.
        - May include definitions, epidemiological data, treatment overview, or references to clinical guidelines.
        - A clinical context belongs to all contraindication coming after, so the clinical context should be assigned to all related contraindications.
      

  action:
  Extract all contraindications of {LCD_title} from {LCD}.
  Return the extracted contraindications as a **valid JSON array of objects**. Don't return any limitation. Each object should follow this format:

    [{{'contraindication ID': create an ID like this Cont_1,
        'contraindication': put the contraindication here,
        'explicit': put "True" if the contraindication is explicit; otherwise put "False"
        'reason': provide a short explanation for implicit contraindications why you picked that statement as a contraindication. For
        explicit contraindications just put 'None'.
        'clinical_context': put `clinical context` here. If there is no clinical context, just put `None`.
    }}]
    

  Use **double quotes** for all keys and string values. Use `True`, `False`, and `Null` for boolean and null values.
  Do not wrap the output in triple backticks or return it as a string. Just return the raw JSON array.

  Each object should include:
  - "contraindication_ID"
  - "contraindication"
  - "explicit"
  - "reason"
  - "clinical_context"

  Use valid JSON syntax with double quotes and JSON-native values (True, False, Null). Do not return a raw list or wrap the output in markdown.

        """

  explicit_mdgs_extraction_prompt = """ 
  thought:
  "Explicit medical guidelines" are typically presented in structured formats such as:
  - Bullet points (<ul><li>) or (&bull;)
  - Numbered or alphabetically ordered lists (<ol type="1"|"A"|"a">)
  - Nested lists of any depth, including combinations of <ul> and <ol> tags.

  These structures may contain multiple levels of nesting, where:
  - A top-level <li> may contain a nested <ol> or <ul>,
  - Each nested list may contain further <li> elements, and so on.

  The HTML may include a mix of list types (e.g., numeric, alphabetic, symbolic), and the nesting may go beyond three levels.

  action:
  Extract all **explicit** medical indications and limitations from {LCD}, a Local Coverage Determination (LCD) document in HTML format.

  Your task must:
  - Traverse all <li> elements, including those nested within <ul> or <ol> tags of any depth.
  - Recursively extract content from nested lists, preserving the full hierarchical structure.
  - Include all structured content, even if it appears under <ol type="A">, <ol type="a">, <ul style="list-style-type: circle;">, or similar.
  - Represent each guideline as a dictionary. If a guideline contains sub-guidelines (e.g., diagnostic criteria, symptoms, or conditions), include them as a nested list under a key called "sub_mdgs". Each sub-guideline should follow the same dictionary format:

              mdg_ID: Use hierarchical IDs like mdg_7.1, mdg_7.2, etc.
              mdg: Text of the sub-guideline.
              type: Same as parent unless specified.
              explicit: put True
              justification: put None
              extraction time: Same as parent.
  - Ignore narrative text outside of structured list elements unless it introduces a list that follows.
      Do not extract introductory or transitional phrases that precede a list (e.g., “will be considered medically necessary in the following circumstances” or “demonstrated by:”).
  - Only extract content that is within <li> elements or clearly structured as a guideline, not standalone headers or lead-ins.
  - If a sentence introduces a list, ignore it unless it is part of a <li> element.

  response format:
  Return the list of medical guidelines, including indications and limitations, without any extra explanation. Each guideline should be organized in a dictionary format with the following keys:
  - mdg_ID: Use sequential IDs like mdg_1, mdg_2, etc.
  - mdg: The extracted medical guideline exactly as it appears in the LCD document.
  - type: Either "indication" or "limitation".
  - explicit: True
  - justification: None
  - extraction time: Current date and time in the format YYYY-MM-DD HH:MM:SS.


  Use **double quotes** for all keys and string values. Use `True`, `False`, and `Null` for boolean and null values.
  Do not wrap the output in triple backticks or return it as a string. Just return the raw JSON array.

  observation:
  Here are examples of explicit indications and limitations from an LCD in HTML format:
  - <li>anti-inflammatory medications or analgesics, or</li>
  - • Loosening of one or both components; or<br /><br />•
  - Multi-level example:
    <li>Chronic migraine is defined as... Treatment of chronic migraines will be covered when they meet the following diagnostic criteria:
      <ol type="A">
        <li><strong>Migraine with aura:</strong>
          <ol>
            <li>At least two attacks fulfilling the following criteria:
              <ol type="a">
                <li><strong>One or more of the following reversible aura symptoms:</strong>
                  <ul>
                    <li>Visual (aura, changes in vision)</li>
                    <li>Sensory (e.g., tingling in hands or face)</li>
                    ...
                  </ul>
                </li>
              </ol>
            </li>
          </ol>
        </li>
      </ol>
    </li>


Do not include any introductory text, explanation, or markdown formatting. Just return the raw JSON array.
          """          
              
  implicit_mdgs_extraction_prompt = """ 

  reminder:  Indications are the clinical scenarios, diagnoses, or patient conditions under which a service, procedure, or item is considered reasonable and necessary and therefore eligible for Medicare coverage.

  thought:
          Implicit indications, refer to those that are not explicitly listed in structured formats such as bullet points, numbered lists, or tables. Instead, they are embedded within narrative paragraphs of the document, often requiring careful interpretation to extract relevant clinical meaning.

          Indications are considered implicit only if there is no semantically identical or equivalent explicit indication listed in the structured\
        sections that follow the paragraph. If a similar indication is later presented explicitly, the narrative mention is treated as contextual\
            reinforcement rather than a distinct implicit rule.,so you shuldn't consider it as an implicit indication.


      For implicit guidelines, the sentence or paragraph, that the implicit guideline is taken from, is considered as 'clinical guideline'
      
  action:
  Your task is to extract implicit medical indications and limitations from {LCD}, which is a Local Coverage Determination (LCD) document in html format. 
  you should respond to this prompt with a list of medical guidelines without any extra explanation.\
  Each indication and limitation should be organized in a dictionary format with the following keys:
      - mdg_ID: put an ID like mdg_1 and mdg_2, etc.,
      - mdg: put here the extracted indication exactly as it is in the LCD document,
      - type: put the type of the medical guideline here as it is indication or limitation.
      - explicit: put False if the medical guideline is implicit, otherwise put True.
      - justification: Please provide a brief justification why you considered this indication as an implicit indication.
      - extraction time: put the current date and time in the format YYYY-MM-DD HH:MM:SS.

  Use **double quotes** for all keys and string values. Use true`, `false`, and `null` for boolean and null values.
  Do not wrap the output in triple backticks or return it as a string. Just return the raw JSON array.


  observation:
  Here is a paragraph with a set of explicit indications afterward:
  <p>In some circumstances, for example, if the patient has bone on bone articulation, severe deformity, pain or significant disabling interference with activities of daily living, the surgeon may determine that nonsurgical medical management would be ineffective or counterproductive and that the best treatment option, after explaining the risks, is surgical. If medical management is deemed appropriate, the medical record should indicate the rationale for and the circumstances under which this is the case.<br /><br /></p>
  <ul>
  <li>Malignancy of the joint involving the bones or soft tissues of the pelvis or proximal femur; <strong> or</strong></li>
  </ul>
  <ul>
  <li>Avascular necrosis (osteonecrosis of femoral head); <strong> or</strong></li>
  </ul>
  <ul>
  <li>Fracture of the femoral neck; <strong> or</strong></li>
  </ul>
  <ul>
  <li>Acetabular fracture; <strong> or</strong></li>
  </ul>
  <ul>
  <li>Non-union or failure of previous hip fracture surgery; <strong> or</strong></li>
  <li>Mal-union of acetabular or proximal femur fracture</li>
  </ul>

  "bone on bone articulation" is an implicit indications that is presented in the paragraph, but it is not listed in the explicit indications.

Do not include any introductory text, explanation, or markdown formatting. Just return the raw JSON array.
          """          

  consolidating_mdgs_prompt = """
  thought:
  You are consolidating medical guidelines extracted from an LCD document.

  Given the following two sets of extracted guidelines:

  Explicit Guidelines:
  {explicit_mdgs_extracted}

  Implicit Guidelines:
  {implicit_mdgs_extracted}

  action:
  Your task is to merge them into a unified list of medical guidelines in the keys:

  - mdg_ID
  - mdg
  - type
  - explicit
  - justification
  - extraction_time

  Please put None if there is no justification for the justification key.

  When merging implicit guidelines into the list of explicit guidelines, assign each implicit guideline a new mdg_ID that continues the numbering \
      from the last explicit guideline.
  Please use the values if each medical guideline as they are in the Explicit Guidelines and in the implicit guidlines.
  Use **double quotes** for all keys and string values. Use `True`, `False`, `None` and `Null` for boolean and none or null values.
  Do not wrap the output in triple backticks or return it as a string. Just return the raw JSON array.

  """

  structure_extraction_prompt = """
  thought:
  In order to capture the complexity of the hierarchical structure of medical guidelines, you have to consider the definition of specific terms as below:
    - parent groups: Indications and limitations are often organized into distinct groups separated by a paragraph, sentence, or section title.
    - parent_id: it is a unique identifier generated for each parent group. pattern_id is like pg_1, pg_2, and so on.
    - child group:  Within a parent group, there may be standalone medical guidelines as well as nested sets of related items, which we will call 'child groups'. Each child group contains sub-indications or sub-limitations that expand on the parent group’s content.
    - child_group_id: it is a unique identifier generated for each child group. child_group_id is like chg_1, chg_2, and so on.
    - type: refers to thr type of medical guideline which is either indication or limitation. 
    - logical relation: refers to those logical AND's or OR's often exist between items within a parent group and its child groups. These logical connectors are typically stated at the end of indications and limitations. If no logical relation is explicitly stated at the end of an explicit indication or explicit limitation in the LCD document—or if you have considered an implicit indication or implicit limitation, please follow these rules:
              Rule 1: If an item is an indication, assume its logical relation is OR.
              Rule 2: If an item is a limitation, assume its logical relation is AND.
    - mdgs: refer to the list of medical guidelines of a specific parent group. Each medical guideline will be represented as a dictionary whose keys are member_id, and content.
    - member_id: is a unique identifer which is created based on parent_id. For example, for pg_1, the identifer of parent group 1, member_id's are mdg_1.1, mdg_2.2, and so on. Likewise, for pg_2, the identifier of parent group 2, member_id's are mdg_2.1, mdg_2.2, and so on.
    - content: is the body of a medical guideline extracted from the LCD.
    - Alongside extracting explicit guidelines, it is equally important to identify the clinical context.

    - clinical context: clinical context typically:
        -- Provides background, rationale, or framing for the indications that follow
        -- Appears as plain text or paragraph(s), or as a title section before explicit guidelines
        -- May include definitions, epidemiological data, treatment overview, or references to clinical guidelines
        -- A clinical context belongs to all explicit medical guidelines that follow it
        
        Context Inheritance Rules:
        -- If a guideline is nested within another (i.e., a sub-guideline), its clinical context is defined by its parent guideline at the higher level of the hierarchy.
        -- If a set of medical guidelines are identified as sub-guidelines listed beneath another guideline, referred to as the parent medical guideline, then:
          --- Each sub-guideline should inherit the parent medical guideline itself as its clinical context.
          --- This means the parent guideline serves as the framing or rationale for all its nested sub-guidelines, regardless of depth.
        --  If there is no clinical context, just put `None`.
        -- Example:
        Advanced joint disease demonstrated by:
            • Radiographic supported evidence...
            •  Pain that cannot be adequately controlled...
            • If appropriate, history of unsuccessful conservative therapy... -- anti-inflammatory medications or analgesics, or
              - flexibility and muscle strengthening exercises, or
              - `Advanced joint disease demonstrated by:` is the clinical context for the three top-level guidelines.
            
            Example Explanation: The two nested guidelines under the third bullet point inherit their clinical context from their parent guideline, which is:
            `If appropriate, history of unsuccessful conservative therapy...`

              
  action:
  First please create a list of indications and limitations from {consolidated_mdgs} called consolidated mdg's in this prompt. The consolidated mdg's may contain multiple levels of nesting. The list should include all entries from every level, capturing the following fields: mdg_ID, mdg, and type. If a guideline is nested under another (e.g., bullet points under a parent bullet), treat it as part of a child group. Use hierarchical identifiers like mdg_1.1, mdg_1.1.1, etc., to reflect nesting depth.
  In the next step, given the mdg's in the created list, look at {LCD}, and according to the above-mentioned thought, organize each parent group of indications or limitations and their probable child groups using the dictionary structure below

    [{{
      'parent_group': {{
        'parent_id': '',
        'type': '',
        'logical_relation': '',
        'mdgs': [
          {{
            'member_id': '',
            'content': '',
            'clinical_context': ''
          }}
        ],

        'child_group': [
          {{
            'child_group_id': '',
            'logical_relation': '',
            'mdgs': [
              {{
                'member_id': '',
                'content': '',
                'clinical_context': ''
              }}
            ],
            child_group": [ ... ]  // Include this key to allow recursive nesting
          }}
        ]
      }}
    }}]

  please note that Each child group may itself contain further nested child groups. Ensure that nesting is preserved recursively.

  All parent groups should be returned as a list and each parent group should be a separate dictionary in the list, and the top-level key in each dictionary must be exactly 'parent_group'.  Do not rename the key to 'parent_group_2', 'parent_group_3', etc.—keep it constant.
  If there are multiple parent groups, return them as the Python list of dictionaries, each following the same structure. 
  Use **double quotes** for all keys and string values. Use `true`, `false`, `none` and `null` for boolean and none or null values.
  Return only a raw JSON array. Do not include any markdown formatting, triple backticks, or explanatory text. Output must be valid JSON only.  

"""


  # Define the prompts
  prompt_template_instruct = PromptTemplate(template=instruct_template, input_variables=[])
  prompt_template_contraindications_extraction = PromptTemplate(template=contraindications_extraction_prompt, input_variables=["LCD_title","LCD"])
  prompt_template_explicit_mdgs_extraction = PromptTemplate(template=explicit_mdgs_extraction_prompt, input_variables=["LCD"])
  prompt_template_implicit_mdgs_extraction = PromptTemplate(template=implicit_mdgs_extraction_prompt, input_variables=["LCD"])
  prompt_template_consolidating_mdgs = PromptTemplate(template=consolidating_mdgs_prompt, input_variables=["explicit_mdgs_extracted", "implicit_mdgs_extracted"])
  prompt_template_structure_extraction = PromptTemplate(template=structure_extraction_prompt, input_variables=["consolidated_mdgs"])

  # Define the chains using the pipe syntax
  chain_instruct = prompt_template_instruct | llm
  chain_contraindications_extraction = prompt_template_contraindications_extraction | llm
  chain_explicit_mdgs_extraction = prompt_template_explicit_mdgs_extraction | llm
  chain_implicit_mdgs_extraction = prompt_template_implicit_mdgs_extraction | llm
  chain_consolidated_mdgs = prompt_template_consolidating_mdgs | llm
  chain_structure_extraction = prompt_template_structure_extraction | llm


  # Ensure inputs are passed as dictionaries with the required keys
  result_instruct = chain_instruct.invoke({})
  result_contraindications = chain_contraindications_extraction.invoke({"LCD_title":LCD_title,"LCD":LCD})
  result_explicit = chain_explicit_mdgs_extraction.invoke({"LCD": LCD})
  result_implicit = chain_implicit_mdgs_extraction.invoke({"LCD": LCD})
  result_consolidated = chain_consolidated_mdgs.invoke({
      "explicit_mdgs_extracted": result_explicit.content,
      "implicit_mdgs_extracted": result_implicit.content
  })
  result_structure_extraction = chain_structure_extraction.invoke({"consolidated_mdgs":result_consolidated.content,"LCD":LCD})

  return extract_json_string(result_contraindications.content), extract_json_string(result_consolidated.content), extract_json_string(result_structure_extraction.content)


In [16]:
contraindications, LCD_mdgs, LCD_decision_tree = extract_LCD_medicalguidelines(LCD, LCD_title)

In [17]:
contraindications

[{'contraindication_ID': 'Cont_1',
  'contraindication': 'Treatment of skin wrinkles using botulinum toxin is cosmetic and is not covered by Medicare.',
  'explicit': True,
  'reason': None,
  'clinical_context': 'Botulinum toxin injections are used to treat various focal muscle spastic disorders and excessive muscle contractions such as dystonia, spasms, twitches, etc.'},
 {'contraindication_ID': 'Cont_2',
  'contraindication': 'Acceptance of botulinum toxin has not been established for deviations over 50 prism diopters.',
  'explicit': True,
  'reason': None,
  'clinical_context': 'Botulinum toxin injections are used to treat various focal muscle spastic disorders and excessive muscle contractions such as dystonia, spasms, twitches, etc.'},
 {'contraindication_ID': 'Cont_3',
  'contraindication': 'Acceptance of botulinum toxin has not been established for restrictive strabismus.',
  'explicit': True,
  'reason': None,
  'clinical_context': 'Botulinum toxin injections are used to trea

In [18]:
LCD_mdgs

[{'mdg_ID': 'mdg_1',
  'mdg': 'Before consideration of coverage may be made, it should be established that the patient has been unresponsive to conventional methods of treatments such as medication, physical therapy and other appropriate methods used to control and/or treat spastic conditions. An exception to this general rule is that for certain treatments including focal dystonia, hemifacial spasm, orofacial dyskinesia, blepharospasm, severe writer’s cramp, laryngeal spasm, or dysphonia, botulinum toxin can be an initial mode of therapy, and in these circumstances, it is not necessary to show that other methods of treatment have been tried and proven unsuccessful.',
  'type': 'indication',
  'explicit': True,
  'justification': None,
  'extraction_time': '2023-10-05 12:00:00'},
 {'mdg_ID': 'mdg_2',
  'mdg': 'Coverage of botulinum toxin for certain spastic conditions (e.g., cerebral palsy, stroke, head trauma, spinal cord injuries and multiple sclerosis) will be limited to those condi

In [19]:
LCD_decision_tree

[{'parent_group': {'parent_id': 'pg_1',
   'type': 'indication',
   'logical_relation': 'or',
   'mdgs': [{'member_id': 'mdg_1',
     'content': 'Before consideration of coverage may be made, it should be established that the patient has been unresponsive to conventional methods of treatments such as medication, physical therapy and other appropriate methods used to control and/or treat spastic conditions. An exception to this general rule is that for certain treatments including focal dystonia, hemifacial spasm, orofacial dyskinesia, blepharospasm, severe writer’s cramp, laryngeal spasm, or dysphonia, botulinum toxin can be an initial mode of therapy, and in these circumstances, it is not necessary to show that other methods of treatment have been tried and proven unsuccessful.',
     'clinical_context': None},
    {'member_id': 'mdg_3',
     'content': 'Botulinum toxin can be used to reduce spasticity or excessive muscular contractions to relieve pain, to assist in posturing and walk

In [20]:
LCD_leaf_nodes = collect_leaf_nodes(LCD_decision_tree)

In [21]:
LCD_leaf_nodes

[{'member_id': 'mdg_1',
  'content': 'Before consideration of coverage may be made, it should be established that the patient has been unresponsive to conventional methods of treatments such as medication, physical therapy and other appropriate methods used to control and/or treat spastic conditions. An exception to this general rule is that for certain treatments including focal dystonia, hemifacial spasm, orofacial dyskinesia, blepharospasm, severe writer’s cramp, laryngeal spasm, or dysphonia, botulinum toxin can be an initial mode of therapy, and in these circumstances, it is not necessary to show that other methods of treatment have been tried and proven unsuccessful.',
  'clinical_context_chain': []},
 {'member_id': 'mdg_3',
  'content': 'Botulinum toxin can be used to reduce spasticity or excessive muscular contractions to relieve pain, to assist in posturing and walking, to allow better range of motion, to permit better physical therapy, and to reduce severe spasm in order to p

In [22]:
def test_QGen(medical_guideline, clinical_context,medical_doc_title,medical_doc_ID,mdg_id):

    question_generation_prompt = f"""
        Instruction:
        Please review the {medical_guideline}, which represents an indication or limitation extracted from a medical document such as a Local Coverage Determination (LCD) or Milliman Care Guidelines (MCG).
        Your task is to:
        - Generate at least one clear and concise question based on the type of medical guideline. The question should be designed to help determine whether relevant information related to the indication and its clinical context exists in a patient's medical record.
            -- Guideline types include: Lateral, Compound, Context-dependent, or Regular.
        - Enhance the generated question by incorporating all applicable clinical conditions listed in {clinical_context}. When doing so, ignore any structural or instructional phrases such as 'as indicated by', 'when all of the following are present', 'such as', or similar. Only include actual clinical conditions or medically relevant terms in the question.            -- If a clinical context entry does not contain any conditions, ignore it and do not include it in the question enhancement process.
        When generating the question:
            - Treat {medical_guideline} as the core subject.
            - Use {clinical_context} to narrow the domain of the question to relevant clinical conditions.
            - Ensure the final question reflects both the guideline and its associated clinical context.
        
        Thought:
        There are four main types of medical guidelines:

        1. Lateral Medical Guidelines  
        1.1. Definition: Involves an anatomical structure that can be lateralized. These include: eye, hip, knee, limb, shoulder, wrist, ankle, and others.
                        - If any of these anatomical parts are explicitly mentioned in the {medical_guideline} the guideline must be classified as 'lateral'.
                        - The follow-up clause must ask: 'If yes, left or right?' without repeating the anatomical part.
        1.2. Instruction:
                        Apply the principle of literality and include a follow-up clause that asks only for laterality — specifically, use the phrase: 'If yes, left or right?' Do not repeat the anatomical part in the follow-up clause. Only consider anatomical parts relevant to {medical_doc_title}.
        1.3. Example: 
                            Indication: Pain or functional disability from injury due to trauma or arthritis of the joint
                            Question: Does the patient experience pain in the knee or functional disability? If yes, which knee?
                        
                            Indication: Radiographic supported evidence or when conventional radiography is not adequate, magnetic resonance imaging (MRI) and/or computed tomography (CT) (in situations when MRI is non-diagnostic or not able to be performed) supported  evidence (subchondral cysts, subchondral sclerosis, periarticular osteophytes, joint subluxation, joint space narrowing, avascular necrosis)
                            Question:
                                    Has the patient undergone radiological exams such as X-ray, CT scan, or MRI on the knee? If yes, which knee?

        2. Compound Medical Guidelines  
        2.1. Definition: Includes multiple distinct clinical indications, each of which could independently justify medical necessity.  
        2.2. Instruction: perform the following steps:
                            Step 1: Identify and separate each distinct indication.
                            Step 2: Generate a medically relevant yes/no question for each distinct indication.

        2.3. Example:
                            Indication: Active urinary tract or dental infection
                            Question 1: Does the patient have active urinary tract infection?
                            Question 2: Does the patient have active dental infection?


        3. Context-Dependent Medical Guidelines  
        3.1. Definition: Includes a clinical indication that is only valid or relevant when a specific clinical context is met.  
        3.2. Instruction: perform the following:
                            - Identify the core indication.
                            - Extract any dependent clinical context.
                            - Incorporate complementary information already provided (e.g., duration of therapy).
                            - Generate two questions:
                            -- Question 1: one medically relevant yes/no question to the indication in order to see whether relevant information exists in a patient's medical record.
                            -- Question 2: one yes/no question relevant to the context to see if relevant information exists in a patient's medical record.
                        
        3.3. Example: 
                    Indication: supervised physical therapy [Activities of daily living (ADLs) diminished despite completing a plan of care,
                    yes/no Question: Has the patient undergone or completed physical therapy?
                    context Question: Has the patient undergone or completed a plan of care?

        4. Regular Medical Guidelines  
        4.1. Definition: Does not meet the criteria for lateral, compound, or context-dependent types.
        4.2. Instruction: generate only one yes/no question that can be answered by reviewing the medical records of a patient. The question should confirm whether relevant data or documentation exists for that indication.

        4.3. Examples:
                    Indication: Active urinary tract infection
                    Generated Question: Is there documentation of an active urinary tract infection in the medical record?
                    Indication: Use of anti-inflammatory medications
                    Generated Question: Is there evidence in the medical record that the patient is using anti-inflammatory medications?
                    Indication: Supervised physical therapy
                    Generated Question: Has the patient undergone supervised physical therapy as documented in the medical record?

        Action:
        Review `{medical_guideline}` and its clinical context, which are `{clinical_context}` and taken from `{medical_doc_title}`.  
        Follow these steps:
        First, determine the type of the medical guideline. Remember that a medical guideline might be lateral, compound, and context-dependent or a combination of these three types, but when it is regular, it cannot be the other three types.
        Second, determine the correct question generation logic based on the guideline type:
                - If the guideline is regular, it must not be treated as lateral, compound, or context-dependent. Only one question must be generated, even if multiple clinical conditions are present in {clinical_context}. These conditions should be combined into a single question.
                - If the guideline is compound, generate a separate question for each distinct indication.
                - If the guideline is context-dependent, generate one question for the indication and one for the context.
                - If the guideline is lateral, generate one question with a follow-up clause for laterality.
        Also, remember that the questions must:
            - determine whether the necessary information exists in the patient's medical record.
            - be specific to the medical guideline type identified in the previous step.
            - be clear, concise, and directly related to the medical guideline.
            - clearly address **all clinical conditions** listed in {clinical_context}.

        Each question must be returned as a dictionary with the following keys:

        - `"medical_guideline_id"`: {mdg_id} which is a unique identifier.
        - `"generated_question_id"`: A unique identifier for each question that its pattern is like Q_1, Q_2, and so on.
        - `"question"`: The generated question text.
        - `"question_type"`: One or a combination of `"lateral"`, `"compound"`, `"context-dependent"`, or only `"regular"`.
        - `"generation_time"`: The current timestamp in ISO 8601 format (e.g., `"2025-08-11T14:03:22Z"`).
        - `"explanation"`: Explain why you chose the question_type. Please provide full explanations with ALL details.

        ---

        Output Format Instructions:

        - Return a **list of dictionaries**, each representing one question.
        - Use **valid JSON syntax** with double quotes.
        - Do **not** wrap the output in Markdown or code blocks.
        - Do **not** include any explanation, commentary, or extra text.
        """

    system_message = SystemMessage(content=question_generation_prompt)

    user_message = HumanMessage(content=f"""
                    Generate question for the following medical guideline and its clinical context:

                    Medical Guideline: {medical_guideline}
                    Clinical Context: {clinical_context}
                    document Title: {medical_doc_title}
                    document ID: {medical_doc_ID}
                    Medical Guideline ID: {mdg_id}
                                                """)
    response = llm.invoke([system_message, user_message])
    questions_generated = extract_json_string(response.content)

    print(f"reason: {questions_generated[0]['question_type']}")
    print(f"reason: {questions_generated[0]['explanation']}")
    print(f'medical guidelins: {medical_guideline}')
    for i in range(len(questions_generated)):      
        print(questions_generated[i]['question'])  

In [23]:
#Farshad for test
medical_guideline = 'Initial evaluation or staging'
clinical_context = ['Bone neoplasm (benign or malignant) involving spine, known or suspected (eg, chondrosarcoma, chordoma, Ewing sarcoma family of tumors, giant cell tumor of bone, osteosarcoma, metastatic tumors), as indicated by 1 or more of the following:', 'Cancer or neoplasm evaluation, staging, or surveillance needed, as indicated by 1 or more of the following:']
medical_doc_title = 'Cervical Spine MRI'
medical_doc_ID = '123'
mdg_id = 'mg_1.2.2'

In [24]:
test_QGen(medical_guideline, clinical_context,medical_doc_title,medical_doc_ID,mdg_id)

reason: regular
reason: The guideline is classified as regular because it does not involve lateralization, multiple distinct indications, or context-dependent conditions. The question is designed to confirm whether there is documentation of an initial evaluation or staging for the specified bone neoplasms or cancer-related conditions in the patient's medical record.
medical guidelins: Initial evaluation or staging
Is there documentation in the patient's medical record of an initial evaluation or staging for bone neoplasm involving the spine, such as chondrosarcoma, chordoma, Ewing sarcoma family of tumors, giant cell tumor of bone, osteosarcoma, or metastatic tumors, or for cancer or neoplasm evaluation, staging, or surveillance?


# Question Generation Function

In [25]:
def question_generation(leaf_nodes, LCD_title, LCD_ID):

    question_generation_prompt = """
    Instruct: Please review {medical_guideline}, which is an indication or limitation extracted from the Local Coverage Determination (LCD) of {LCD_title}.
            Your task is to generate clear and concise questions that directly address the indication or limitation.
            Also, question should address clinical conditions, known as clinical context, which are seperately collected in a list which is available in {clinical_context}.
            The purpose of a question is to help determine whether relevant information to an indication as well as its clinical context exist in a patient's medical record.
            
    Thought:
    There are four main types of medical guidelines (indications and limitations):

    1. Lateral Medical Guidelines  
    1.1. Definition: Involves an anatomical structure that:  
        - Can be lateralized (e.g., hip, knee, eye, limb), and  
        - Is directly mentioned in {LCD_title}, or  
        - Is part of or related to a condition mentioned in {LCD_title}.
    1.2. Instruction:
    apply the principle of literality and include a follow-up clause such as:
                    'If yes, which [hip/eye/limb/knee]?' Only consider anatomical parts relevant to {LCD_title}.
    1.3. Example: 
                        Indication: Pain or functional disability from injury due to trauma or arthritis of the joint
                        Question: Does the patient experience pain in the knee or functional disability? If yes, which knee?
                    
                        Indication: Radiographic supported evidence or when conventional radiography is not adequate, magnetic resonance imaging (MRI) and/or computed tomography (CT) (in situations when MRI is non-diagnostic or not able to be performed) supported  evidence (subchondral cysts, subchondral sclerosis, periarticular osteophytes, joint subluxation, joint space narrowing, avascular necrosis)
                        Question:
                                Has the patient undergone radiological exams such as X-ray, CT scan, or MRI on the knee? If yes, which knee?

    2. Compound Medical Guidelines  
    2.1. Definition: Includes multiple distinct clinical indications, each of which could independently justify medical necessity.  
    2.2. Instruction: perform the following steps:
                        Step 1: Identify and separate each distinct indication.
                        Step 2: Generate a medically relevant yes/no question for each distinct indication.

    2.3. Example:
                        Indication: Active urinary tract or dental infection
                        Question 1: Does the patient have active urinary tract infection?
                        Question 2: Does the patient have active dental infection?


    3. Context-Dependent Medical Guidelines  
    3.1. Definition: Includes a clinical indication that is only valid or relevant when a specific clinical context is met.  
    3.2. Instruction: perform the following:
                        - Identify the core indication.
                        - Extract any dependent clinical context.
                        - Incorporate complementary information already provided (e.g., duration of therapy).
                        - Generate two questions:
                         -- Question 1: one medically relevant yes/no question to the indication in order to see whether relevant information exists in a patient's medical record.
                         -- Question 2: one yes/no question relevant to the context to see if relevant information exists in a patient's medical record.
                     
    3.3. Example: 
                Indication: supervised physical therapy [Activities of daily living (ADLs) diminished despite completing a plan of care,
                yes/no Question: Has the patient undergone or completed physical therapy?
                context Question: Has the patient undergone or completed physical therapy?

    4. Regular Medical Guidelines  
    4.1. Definition: Does not meet the criteria for lateral, compound, or context-dependent types.
    4.2. Instruction: generate a yes/no question that can be answered by reviewing the medical records of a patient. The question should confirm whether relevant data or documentation exists for that indication.

    4.3. Examples:
                Indication: Active urinary tract infection
                Generated Question: Is there documentation of an active urinary tract infection in the medical record?
                Indication: Use of anti-inflammatory medications
                Generated Question: Is there evidence in the medical record that the patient is using anti-inflammatory medications?
                Indication: Supervised physical therapy
                Generated Question: Has the patient undergone supervised physical therapy as documented in the medical record?

    Action:
    Review `{medical_guideline}` and its clinical context, which are `{clinical_context}` and taken from `{LCD_title}`.  
    Follow these steps:
    First, determine the type of the medical guideline. Remember that a medical guideline might be lateral, compound, and context-dependent or a combination of these three types, but when it is regular, it cannot be the other three types.
    Second, according to the type or types of the medical guideline, follow the related instructions questions generation.
     Also, remember that the questions must:
        - determine whether the necessary information exists in the patient's medical record.
        - be specific to the medical guideline type identified in the previous step.
        - be clear, concise, and directly related to the medical guideline.
        - clearly address **all clinical conditions** listed in {clinical_context}.

    Each question must be returned as a dictionary with the following keys:

    - `"medical_guideline_id"`: {mdg_id} which is a unique identifier.
    - `"generated_question_id"`: A unique identifier for each question that its pattern is like Q_1, Q_2, and so on.
    - `"question"`: The generated question text.
    - `"question_type"`: One or a combination of `"lateral"`, `"compound"`, `"context-dependent"`, or only `"regular"`.
    - `"generation_time"`: The current timestamp in ISO 8601 format (e.g., `"2025-08-11T14:03:22Z"`).

    ---

    Output Format Instructions:

    - Return a **list of dictionaries**, each representing one question.
    - Use **valid JSON syntax** with double quotes.
    - Do **not** wrap the output in Markdown or code blocks.
    - Do **not** include any explanation, commentary, or extra text.
    """
    
    
    system_message = SystemMessage(content=question_generation_prompt)


    questions_df = pd.DataFrame(columns=['LCD_title','MDG_ID', 'Medical Guideline','Medical Guideline Type','Clinical Context', 'question_id','Question'])
    
    for leaf in leaf_nodes:
        mdg_id = leaf['member_id']
        medical_guideline = leaf['content']
        clinical_context = leaf['clinical_context_chain']
    
        
        user_message = HumanMessage(content=f"""
                Generate question for the following medical guideline and its clinical context:

                Medical Guideline: {medical_guideline}
                Clinical Context: {clinical_context}
                LCD Title: {LCD_title}
                LCD ID: {LCD_ID}
                Medical Guideline ID: {mdg_id}
                                            """)

        

        response = llm.invoke([system_message, user_message])
        questions_generated = extract_json_string(response.content)
                
        
        for i in range(len(questions_generated)):                          
            new_row = {
                'LCD_title': LCD_title,
                'MDG_ID': mdg_id,
                'Medical Guideline': medical_guideline,
                'Medical Guideline Type': questions_generated[i]['question_type'],
                'Clinical Context': clinical_context,
                'question_id': questions_generated[i]['generated_question_id'],
                'Question': questions_generated[i]['question']
            }

            questions_df = pd.concat([questions_df,pd.DataFrame([new_row])], ignore_index=True)
    


    # add question_id to the database       
    questions_df["question_id"] = ["Q_" + str(i) for i in range(1, len(questions_df) + 1)]

    # Move 'question_id' to the first column
    cols = ["question_id"] + [col for col in questions_df.columns if col != "question_id"]
    questions_df = questions_df[cols]

    return questions_df

In [26]:
LCD_questions_df = question_generation(LCD_leaf_nodes, LCD_title, LCD_ID)
LCD_questions_df

,question_id,LCD_title,MDG_ID,Medical Guideline,Medical Guideline Type,Clinical Context,Question
0,Q_1,Botulinum Toxin Types A and B,mdg_1,"Before consideration of coverage may be made, ...",compound,[],Has the patient been unresponsive to conventio...
1,Q_2,Botulinum Toxin Types A and B,mdg_1,"Before consideration of coverage may be made, ...",compound,[],Is the patient being considered for botulinum ...
2,Q_3,Botulinum Toxin Types A and B,mdg_3,Botulinum toxin can be used to reduce spastici...,regular,[],Is there documentation in the medical record t...
3,Q_4,Botulinum Toxin Types A and B,mdg_4,Botulinum toxin has indications for overactive...,compound,[],Is there documentation in the medical record i...
4,Q_5,Botulinum Toxin Types A and B,mdg_4,Botulinum toxin has indications for overactive...,compound,[],Is there documentation in the medical record i...
5,Q_6,Botulinum Toxin Types A and B,mdg_6,There may be patients who require Electromyogr...,regular,[],Has the patient undergone Electromyography (EM...
6,Q_7,Botulinum Toxin Types A and B,mdg_7,For the appropriate initial and total doses of...,regular,[],Is there documentation in the medical record t...
7,Q_8,Botulinum Toxin Types A and B,mdg_12,Botulinum Toxin is covered for prophylaxis of ...,compound,[],Is there documentation in the medical record t...
8,Q_9,Botulinum Toxin Types A and B,mdg_12,Botulinum Toxin is covered for prophylaxis of ...,compound,[],Is there evidence in the medical record that t...
9,Q_10,Botulinum Toxin Types A and B,mdg_11.1,The patient has failed conventional therapy.,context-dependent,"[For treatment of achalasia and cardio spasm, ...",Has the patient failed conventional therapy fo...


In [27]:
print(LCD_questions_df.shape)
print(LCD_questions_df.columns)

(36, 7)
Index(['question_id', 'LCD_title', 'MDG_ID', 'Medical Guideline',
       'Medical Guideline Type', 'Clinical Context', 'Question'],
      dtype='object')


# Question Evalution Function

In [28]:
def questions_evaluation(questions_df):
    
    evaluation_df = pd.DataFrame(columns = ['question_id', 'relevancy', 'relevancy_justification', 'accuracy', 'accuracy_justification', 'clarity', 'clarity_justification', 'completeness', 'completeness_justification'])
    
    question_evaluation_prompt = ''' Please evaluate this question, {question}, to see how much it is aligned to {medical_guideline} and all {clinical_context}, based on the below criteria:
    1. Relevancy
    Definition: Measures how directly the question relates to the specific medical guideline being referenced. A highly relevant question will reflect the intent, scope, and clinical context of the guideline.
    Why it matters: Ensures the question is grounded in the correct clinical framework and supports guideline-based decision-making.
    2. Accuracy
    Definition: Assesses whether the question uses correct medical terminology, reflects current clinical standards, and avoids factual errors. It should be consistent with established medical knowledge and policy concepts.
    Why it matters: Prevents misinterpretation and ensures safe, evidence-based care.
    3. Clarity
    Definition: Evaluates whether the question is clearly worded and interpretable in only one way. It should avoid vague language, double meanings, or overly complex phrasing.
    Why it matters: Reduces confusion and ensures consistent understanding across reviewers or systems.
    4. Completeness
    Definition: Determines whether the question includes all necessary qualifiers, conditions, and context to make a guideline-based decision. It should not omit critical information that could affect the outcome.
    Why it matters: Supports comprehensive evaluation and minimizes the risk of incorrect or incomplete decisions.

    Please use this rubric to score the criteria:
    0: 	Completely unrelated to the guideline; no identifiable connection.
    0.25: 	Minimally related; vague or tangential reference to the guideline.
    0.5: 	Partially related; some relevant elements but lacks direct alignment.
    0.75:	Mostly related; aligns with the guideline but misses some nuance or scope.
    1:	Fully aligned; clearly reflects the intent, scope, and clinical context of the guideline.

    In the end, please return the result in the form of this sictionary:
    {{'question_id': {question_id},
      'question': question,
      'relevancy': relevancy_score,
      'relevancy_justification': explain your justification about relevancy score here,
      'accuracy': accuracy_score,
      'accuracy_justification': explain your justification about accuracy score here,
      'clarity':clarity_score,
      'clarity_justification': explain your justification about clarity score here,
      'completeness': Completeness_score
      'completeness_justification': explain your justification about completeness score here,
    }}

   
    output format:
    - Return only as **valid JSON syntax** with double quotes 
    - Do **not** wrap the output in Markdown or code blocks.
    - Do **not** include any explanation, commentary, or extra text.
    '''
    
    system_message = SystemMessage(content= question_evaluation_prompt)

    for _, row in questions_df.iterrows():
        question_id = row['question_id']
        question = row['Question']
        medical_guideline = row['Medical Guideline']
        clinical_context = row['Clinical Context']
        
       
        user_message = HumanMessage(content=f"""
                        Evaluate the following question based on the rubric:

                        question_id: {question_id}
                        Question: {question}
                        Medical Guideline: {medical_guideline}
                        Clinical Context: {clinical_context}
                        """)


        # Call the LLM
        response = llm.invoke([system_message, user_message])
        scores = extract_json_string(response.content)

        del scores['question']
        
        new_df = pd.DataFrame([scores])
        # Only concatenate if `scores` is not empty or all-NA
        if not new_df.isna().all(axis=1).all():
            evaluation_df = pd.concat([evaluation_df, new_df], ignore_index=True)

    
    evaluation_df['confidence_score'] = (evaluation_df.relevancy+evaluation_df.accuracy+evaluation_df.clarity+evaluation_df.completeness)/4
    evaluation_df['justifications_summary'] = [get_completion(f'summarize {row.relevancy_justification} and {row.accuracy_justification} and {row.clarity_justification} and {row.completeness_justification}')
        for _,row in evaluation_df.iterrows()
        ]
    
    evaluation_df['confidence_level'] = ['High' if score >= 0.75 else
                                         'Medium' if score >= 0.5 else
                                         'Low'
                                        for score in evaluation_df['confidence_score']
                                        ]


    return evaluation_df

# Question Generation Evaluation DataFrame

In [29]:
LCD_evaluation_df = questions_evaluation(LCD_questions_df)

In [30]:
LCD_evaluation_df

,question_id,relevancy,relevancy_justification,accuracy,accuracy_justification,clarity,clarity_justification,completeness,completeness_justification,confidence_score,justifications_summary,confidence_level
0,Q_1,1,The question directly aligns with the guidelin...,1,The question uses correct medical terminology ...,1,The question is clearly worded and interpretab...,0.75,The question is mostly complete but does not m...,0.9375,The question effectively aligns with the guide...,High
1,Q_2,1,The question directly addresses the guideline'...,1,The question uses correct medical terminology ...,1,The question is clearly worded and interpretab...,1.00,The question includes all necessary qualifiers...,1.0,The question effectively addresses the guideli...,High
2,Q_3,0.75,The question is mostly related to the guidelin...,1,The question accurately uses medical terminolo...,1,The question is clearly worded and interpretab...,0.50,The question is partially complete as it focus...,0.8125,The question primarily addresses the use of bo...,High
3,Q_4,1,The question is fully aligned with the medical...,1,The question accurately uses medical terminolo...,1,The question is clearly worded and interpretab...,1.00,The question includes all necessary qualifiers...,1.0,The question is well-aligned with medical guid...,High
4,Q_5,1,The question is fully aligned with the medical...,1,The question accurately uses medical terminolo...,1,The question is clearly worded and interpretab...,1.00,The question includes all necessary qualifiers...,1.0,The question is well-aligned with medical guid...,High
5,Q_6,1,The question is fully aligned with the medical...,1,The question uses correct medical terminology ...,1,The question is clearly worded and interpretab...,1.00,The question includes all necessary qualifiers...,1.0,The question is well-aligned with the medical ...,High
6,Q_7,1,The question is fully aligned with the medical...,1,The question uses correct medical terminology ...,1,The question is clearly worded and interpretab...,1.00,The question includes all necessary qualifiers...,1.0,The question is well-aligned with medical guid...,High
7,Q_8,1,The question is fully aligned with the medical...,1,The question accurately uses medical terminolo...,1,The question is clearly worded and interpretab...,1.00,The question includes all necessary qualifiers...,1.0,The question is fully aligned with medical gui...,High
8,Q_9,1,The question is fully aligned with the medical...,1,The question uses correct medical terminology ...,1,The question is clearly worded and interpretab...,1.00,The question includes all necessary qualifiers...,1.0,The question is well-aligned with medical guid...,High
9,Q_10,1,The question is fully aligned with the guideli...,1,The question uses correct medical terminology ...,1,The question is clearly worded and interpretab...,1.00,The question includes all necessary qualifiers...,1.0,The question is well-constructed and adheres t...,High


In [31]:
#LCD_mds view just reading data
def transform_mdgs(data):
    mdg_list =data
    # Function to extract relevant fields from an mdg entry
    def extract_fields(mdg):
        return {
            'member_id': mdg.get('mdg_ID'),
            'content': mdg.get('mdg'),
            'type': mdg.get('type'),
            'justification': mdg.get('justification'),
            'category': mdg.get('explicit')
        }

    # Flatten the list including sub_mdgs if present
    all_mdgs = []
    for mdg in mdg_list:
        all_mdgs.append(extract_fields(mdg))
        if 'sub_mdgs' in mdg:
            for sub_mdg in mdg['sub_mdgs']:
                all_mdgs.append(extract_fields(sub_mdg))

    # Convert to pandas DataFrame
    df = pd.DataFrame(all_mdgs)
    return df

# Example usage
#LCD_mdgs
df_mdgs = transform_mdgs(LCD_mdgs)
df_mdgs


,member_id,content,type,justification,category
0,mdg_1,"Before consideration of coverage may be made, ...",indication,None,True
1,mdg_2,Coverage of botulinum toxin for certain spasti...,limitation,None,True
2,mdg_3,Botulinum toxin can be used to reduce spastici...,indication,None,True
3,mdg_4,Botulinum toxin has indications for overactive...,indication,None,True
4,mdg_5,Due to the rarity of severe organic writer's c...,limitation,None,True
5,mdg_6,There may be patients who require Electromyogr...,indication,None,True
6,mdg_7,For the appropriate initial and total doses of...,indication,None,True
7,mdg_8,Coverage of treatments provided may be continu...,limitation,None,True
8,mdg_9,Requests may be considered for redetermination...,limitation,None,True
9,mdg_10,Medicare will allow payment for one injection ...,limitation,None,True


In [32]:
print(df_mdgs.shape)

(31, 5)


In [33]:
def transform_decision_tree(tree_data, questions_df):
    data = tree_data 
    # Initialize a list to collect all rows
    rows = []
    # Recursive function to extract mdgs from any group level
    def extract_mdgs(group, data_type, logical_relation, child_group_id=None, context=None):
        for mdg in group.get('mdgs', []):
            rows.append({
                'data_type': data_type,
                'logical_relation': logical_relation,
                'child_group_id': child_group_id,
                'member_id': mdg.get('member_id'),
                'content': mdg.get('content'),
                'clinical_context': mdg.get('clinical_context') if context is None else context
            })
        for child in group.get('child_group', []):
            extract_mdgs(child, data_type, child.get('logical_relation'), child.get('child_group_id'), context)

    # Traverse each parent group
    for entry in data:
        parent = entry.get('parent_group', {})
        context = parent['mdgs'][0].get('clinical_context') if parent.get('mdgs') else None
        extract_mdgs(parent, parent.get('type'), parent.get('logical_relation'), None, context)

    # Convert to DataFrame
    dession_tree_df = pd.DataFrame(rows)


    #Merging data and making Equivalent for Question_df
    dession_tree_result = pd.merge(dession_tree_df, questions_df, left_on='member_id', right_on='MDG_ID', how='left')
    print(dession_tree_result.columns)
    # Select only the columns you need
    final_dession_tree_df = dession_tree_result[['member_id', 'logical_relation','content','data_type', 'Question','Medical Guideline Type']]
    
    # For database insertion  Nan value change to empty
    final_dession_tree_df = final_dession_tree_df.fillna({
        'logical_relation': '',
        'content': '',
        'Question': '',
        'Medical Guideline Type': ''
    })

    final_dession_tree_df.rename(columns={'data_type': 'classification_type'}, inplace=True)
    final_dession_tree_df.rename(columns={'Question': 'question'}, inplace=True)
    final_dession_tree_df.rename(columns={'Medical Guideline Type': ' medical_guideline_type'}, inplace=True)
   
    return final_dession_tree_df


In [34]:
final_dession_tree_df = transform_decision_tree(LCD_decision_tree, LCD_questions_df)
final_dession_tree_df

Index(['data_type', 'logical_relation', 'child_group_id', 'member_id',
       'content', 'clinical_context', 'question_id', 'LCD_title', 'MDG_ID',
       'Medical Guideline', 'Medical Guideline Type', 'Clinical Context',
       'Question'],
      dtype='object')


,member_id,logical_relation,content,classification_type,question,medical_guideline_type
0,mdg_1,or,"Before consideration of coverage may be made, ...",indication,Has the patient been unresponsive to conventio...,compound
1,mdg_1,or,"Before consideration of coverage may be made, ...",indication,Is the patient being considered for botulinum ...,compound
2,mdg_3,or,Botulinum toxin can be used to reduce spastici...,indication,Is there documentation in the medical record t...,regular
3,mdg_4,or,Botulinum toxin has indications for overactive...,indication,Is there documentation in the medical record i...,compound
4,mdg_4,or,Botulinum toxin has indications for overactive...,indication,Is there documentation in the medical record i...,compound
5,mdg_6,or,There may be patients who require Electromyogr...,indication,Has the patient undergone Electromyography (EM...,regular
6,mdg_7,or,For the appropriate initial and total doses of...,indication,Is there documentation in the medical record t...,regular
7,mdg_11,or,"For treatment of achalasia and cardio spasm, b...",indication,,
8,mdg_12,or,Botulinum Toxin is covered for prophylaxis of ...,indication,Is there documentation in the medical record t...,compound
9,mdg_12,or,Botulinum Toxin is covered for prophylaxis of ...,indication,Is there evidence in the medical record that t...,compound


In [35]:
from typing import List, Dict, Any, Optional, Tuple
import pandas as pd

def _parse_segments(member_id: str) -> Tuple:
    """
    Turn 'mdg_1.10.2' -> (1, 10, 2) for natural numeric sort.
    If non-numeric tokens appear, fall back to string order for that token.
    """
    if '_' in member_id:
        base = member_id.split('_', 1)[1]
    else:
        base = member_id
    parts = base.split('.') if base else []
    segs = []
    for p in parts:
        p = p.strip()
        if p.isdigit():
            segs.append(int(p))
        else:
            try:
                segs.append(int(p))
            except Exception:
                segs.append(p)
    return tuple(segs)

def _parent_of(member_id: str) -> Optional[str]:
    """
    Parent by trimming after last dot; 'mdg_1' is root → None.
    """
    if '_' in member_id:
        prefix, rest = member_id.split('_', 1)
        if '.' not in rest:
            return None
        parent_rest = rest.rsplit('.', 1)[0]
        return f"{prefix}_{parent_rest}"
    return None if '.' not in member_id else member_id.rsplit('.', 1)[0]

def validate_dataframe(df: pd.DataFrame, id_col: str, relation_col: str, content_col: str, question_col:str,classification_type_col:str) -> List[str]:
    """
    Return list of problems; empty list means OK.
    """
    problems = []
    for col in (id_col, relation_col, content_col,question_col,classification_type_col):
        if col not in df.columns:
            problems.append(f"Missing required column: {col}")
    if problems:
        return problems

    invalid_rows = []
    for i, v in enumerate(df[id_col].astype(str).tolist()):
        vv = v.strip()
        if vv == '' or vv.lower() == 'any':
            invalid_rows.append((i, v, 'empty_or_ANY'))
        if '_' not in vv:
            invalid_rows.append((i, v, 'missing_underscore'))
        else:
            after = vv.split('_', 1)[1]
            if after == '' or not any(ch.isdigit() for ch in after):
                invalid_rows.append((i, v, 'no_numeric_part'))
    if invalid_rows:
        details = '; '.join([f"row={i} value={repr(v)} reason={r}" for i, v, r in invalid_rows])
        problems.append(f"Invalid {id_col} detected: {details}")
    return problems

def df_to_hierarchical_json(
    df: pd.DataFrame,
    id_col: str = 'member_id',
    relation_col: str = 'logical_relation',
    content_col: str = 'content',
    question_col:str = 'question',
    classification_type_col: str = 'classification_type',
    seq_col: Optional[str] = None,
    include_seq: bool = True,
    return_single_root: bool = False,
    create_missing_parents: bool = False,
    hide_empty_children: bool = True,    # <-- NEW: remove children key if it’s empty
) -> Any:
    """
    Transform a flat DataFrame with dot-notated master_id into hierarchical JSON.

    - Orders siblings by seq_col if provided, else by natural numeric order of master_id.
    - Optionally include 'seq' in nodes.
    - Optionally create missing parents as stub nodes (content='', logical_relation='ANY').
    - Optionally hide 'children' when empty (leaf nodes).
    """
    problems = validate_dataframe(df, id_col, relation_col, content_col, question_col, classification_type_col)
    if problems:
        raise ValueError("; ".join(problems))

    df_work = df.copy()

    # Determine ordering
    if seq_col and seq_col in df_work.columns:
        df_work = df_work.sort_values(by=seq_col, kind='stable').reset_index(drop=True)
        order_keys = df_work[seq_col].tolist()
    else:
        df_work = df_work.sort_values(by=id_col, key=lambda s: s.astype(str).map(_parse_segments),
                                      kind='stable').reset_index(drop=True)
        order_keys = list(range(len(df_work)))

    # Normalize rows
    rows: List[Dict[str, Any]] = []
    for pos, row in enumerate(df_work[[id_col, relation_col, content_col, question_col,classification_type_col]].itertuples(index=False)):
        identifier, logical_relation, content,question,classification_type = row
        rows.append({
            'seq': order_keys[pos],
            id_col: str(identifier).strip(),
            relation_col: logical_relation,
            content_col: content,
            question_col: question,
            classification_type_col: classification_type
        })

    nodes: Dict[str, Dict[str, Any]] = {}

    # Create nodes
    for r in rows:
        mid = r[id_col]
        if mid not in nodes:
            nodes[mid] = {
                'member_id': mid,
                'content': r[content_col],
                'logical_relation': r[relation_col],
                'question':r[question_col],
                'classification_type':r[classification_type_col],
                'children': []  # will prune later if empty and hide_empty_children=True
            }
            if include_seq:
                nodes[mid]['seq'] = r['seq']
        else:
            # duplicate IDs: keep first content; keep earliest seq
            if include_seq:
                nodes[mid]['seq'] = min(nodes[mid].get('seq', r['seq']), r['seq'])

    # Optionally create any missing parents
    if create_missing_parents:
        ids = list(nodes.keys())
        for mid in ids:
            p = _parent_of(mid)
            while p is not None and p not in nodes:
                nodes[p] = {
                    'member_id': p,
                    'content': '',
                    'logical_relation': 'ANY',
                    'question':'',
                    'classification_type':'',
                    'children': []
                }
                p = _parent_of(p)

    # Attach children
    roots: List[Dict[str, Any]] = []
    for mid, node in nodes.items():
        parent_id = _parent_of(mid)
        if parent_id is None or parent_id not in nodes:
            roots.append(node)
        else:
            nodes[parent_id]['children'].append(node)

    # Sort children (and roots)
    def _sort_children(node: Dict[str, Any]):
        ch = node.get('children', [])
        if not ch:
            return
        if include_seq:
            ch.sort(key=lambda n: n.get('seq', 0))
        else:
            ch.sort(key=lambda n: _parse_segments(n['member_id']))
        for c in ch:
            _sort_children(c)

    for r in roots:
        _sort_children(r)

    if include_seq:
        roots.sort(key=lambda n: n.get('seq', 0))
    else:
        roots.sort(key=lambda n: _parse_segments(n['member_id']))

    # --- NEW: prune empty children keys ---
    def _prune_empty_children(node: Dict[str, Any]):
        if 'children' in node:
            # Recurse first to allow deeper pruning
            for ch in list(node.get('children', [])):
                _prune_empty_children(ch)
            # If children now empty, remove the key
            if not node['children']:
                del node['children']

    if hide_empty_children:
        for root in roots:
            _prune_empty_children(root)

    if return_single_root and len(roots) == 1:
        return roots[0]
    return roots

In [36]:
#Use the Fuction 
df = final_dession_tree_df
# df has columns: master_id, logical_relation, content (and optionally seq)
tree = df_to_hierarchical_json(
    df,
    id_col='member_id',
    relation_col='logical_relation',
    content_col='content',
    question_col='question',
    classification_type_col='classification_type',
    seq_col=None,                 # or 'seq' if you have explicit order
    include_seq=True,
    return_single_root=False,
    create_missing_parents=False,
    hide_empty_children=True      # <-- ensure leaf nodes do not show 'children'
)
print("Out put of Json")
tree

Out put of Json


[{'member_id': 'mdg_1',
  'content': 'Before consideration of coverage may be made, it should be established that the patient has been unresponsive to conventional methods of treatments such as medication, physical therapy and other appropriate methods used to control and/or treat spastic conditions. An exception to this general rule is that for certain treatments including focal dystonia, hemifacial spasm, orofacial dyskinesia, blepharospasm, severe writer’s cramp, laryngeal spasm, or dysphonia, botulinum toxin can be an initial mode of therapy, and in these circumstances, it is not necessary to show that other methods of treatment have been tried and proven unsuccessful.',
  'logical_relation': 'or',
  'question': 'Has the patient been unresponsive to conventional methods of treatment such as medication, physical therapy, and other appropriate methods for controlling and/or treating spastic conditions?',
  'classification_type': 'indication',
  'seq': 0},
 {'member_id': 'mdg_2',
  'c

# Adding Decision tree Structure Criteria content 

In [37]:
#Requestion LLM To Add  short promt to Generate criteria content to json object
add_critera_prompt = PromptTemplate(
    template="You are the medical expert. In the given JSON schema, for every attribute add one field called 'clinical_criteria' in three to four words from the content of same schema level. Please share the updated JSON schema. /n {json}",
    input_variables=['json']
)

json_parser = JsonOutputParser()
json_chain = add_critera_prompt | llm | json_parser
final_json = json_chain.invoke({'json': tree})
print(final_json)

[{'member_id': 'mdg_1', 'content': 'Before consideration of coverage may be made, it should be established that the patient has been unresponsive to conventional methods of treatments such as medication, physical therapy and other appropriate methods used to control and/or treat spastic conditions. An exception to this general rule is that for certain treatments including focal dystonia, hemifacial spasm, orofacial dyskinesia, blepharospasm, severe writer’s cramp, laryngeal spasm, or dysphonia, botulinum toxin can be an initial mode of therapy, and in these circumstances, it is not necessary to show that other methods of treatment have been tried and proven unsuccessful.', 'logical_relation': 'or', 'question': 'Has the patient been unresponsive to conventional methods of treatment such as medication, physical therapy, and other appropriate methods for controlling and/or treating spastic conditions?', 'classification_type': 'indication', 'seq': 0, 'clinical_criteria': 'unresponsive to c

In [38]:
with open('LCD_data_tree.json', 'w') as json_file:
    json.dump(final_json, json_file, indent=4)

# HTML and PDF Generation 

In [93]:
import json
#from weasyprint import HTML
# Load the JSON data from file
with open("LCD_data_tree.json", "r", encoding="utf-8") as file:
    json_data = json.load(file)

# Function to convert JSON to HTML
def html_convert(data):
    def render_node(node):
        html = "<li>"
        #html += f"<div class='node'><span class='member'>{node.get('member_id', '')}</span></div>"
        if  node.get('children'):
             html += f"<div class='logical-section'><div class='item-type'> Type: <strong>{node['classification_type']}</strong></div><div class='item-logic'> Condition Relation: <strong class='logic-relation'>{node['logical_relation']}  </strong></div></div>"
             html += f"<div class='criteria'>{node.get('clinical_criteria', '')}</div>"
             html += f"<div class='content'> {node['content']}</div>"
             
        # Only include clinical_criteria if node has no children
        if not node.get('children'):   
                html += f"<div class='criteria'>{node['clinical_criteria']}</div>"
                html += f"<div class='content'> {node['content']}</div>"
                html += f"<div class='question'><strong>Question:</strong> {node['question']}</div>"

       
        if 'children' in node and node['children']:
            html += "<ul class='children-ul'>"
            for child in node['children']:
                html += render_node(child)
            html += "</ul>"
        html += "</li>"
        
        return html

    html_output = "<ul class='main-ul'>"
    for item in data:
        html_output += render_node(item)
    html_output += "</ul>"
    
    return html_output

# CSS for styling
css_styles = """
<style>
body {
    font-family: Arial, sans-serif;
    font-size: '12px';
}
.logical-section{
 display:flex;
 height:26px;
}
.item-type{
  flex: 0 0 15%; 
}
.item-logic{
flex:0 0 85%
}

.logic-relation{
 font-size:14px;
 text-transform: uppercase;
}
h1 {
    color: #2c3e50;
    text-align:center;
}
ul.main-ul {
    padding-left: 0px;
    list-style:none;
    border:1px solid #d1d1d1;
}
ul.main-ul li {
    margin-bottom: 2px;
    padding: 10px 10px;
    display:block;
}
ul.children-ul {
      border-left:1px solid #d1d1d1;
      border-top:1px solid #d1d1d1;
      list-style: none;
      display:block;
      margin-left:30px;
      margin-top:10px;
      padding:0px;

   }
ul.children-ul > li {   padding: 10px 10px; }

.children-ul::before {
  content: "";
  position: absolute;
  height: 1.5px;
  width:40px;
  margin-left:-40px;
  margin-top:20px;
  background-color: #ccc;
}

li:has(.logical-section) {
  border-top:1px solid #d1d1d1;
  border-bottom:1px solid #d1d1d1;
}

.node {
    font-weight: bold;
    color: #2c3e50;
}
.member {
    color: #2980b9;
}
.question {
    margin-left: 30px;
    color: #8e44ad;
}
.criteria {
    font-weight:bold;
    text-transform: capitalize; 
}
</style>
"""

# Generate full HTML
html_body = html_convert(json_data)
full_html = f"<!DOCTYPE html><html><head>{css_styles}</head><body><h1>{LCD_title}</h1>{html_body}</body></html>"

# Save HTML to file
with open(f"{LCD_title}_output.html", "w", encoding="utf-8") as html_file:
    html_file.write(full_html)

# Convert HTML to PDF
#HTML(string=full_html).write_pdf("LCD_Output.pdf")
#print("Styled HTML and PDF have been successfully generated.")